# Atividade HuggingFace - RAG

**Tema escolhido:** RAG - Retrieval Augmented Generation

Este notebook demonstra uma aplicação simples de perguntas e respostas baseada em uma pequena base de conhecimento. A aplicação busca o texto mais relevante para a pergunta e usa um modelo do HuggingFace para responder com base nesse contexto.

## 1. Instalação das bibliotecas

Execute a célula abaixo no Google Colab ou em um ambiente Jupyter. Se as bibliotecas já estiverem instaladas, você pode pular esta etapa.

In [ ]:
# Execute esta célula no Google Colab ou Jupyter
# No ambiente local, você pode instalar pelo terminal também.

!pip install -q transformers sentence-transformers scikit-learn torch

## 2. Importação das bibliotecas

A aplicação usa:

- `sentence-transformers` para transformar textos em embeddings;
- `sklearn` para calcular similaridade entre pergunta e documentos;
- `transformers` para carregar um modelo de Question Answering do HuggingFace.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import pandas as pd

## 3. Base de conhecimento

Nesta etapa criamos uma pequena base textual. Em um projeto real, essa base poderia vir de PDFs, sites, documentos acadêmicos, banco de dados ou arquivos internos.

In [ ]:
documentos = [
    "RAG, ou Retrieval Augmented Generation, é uma técnica que combina recuperação de informações com geração de respostas por modelos de linguagem.",
    "O HuggingFace é uma plataforma que disponibiliza modelos, datasets e ferramentas para projetos de Inteligência Artificial.",
    "Embeddings são representações numéricas de textos, palavras ou frases, usadas para medir similaridade semântica entre conteúdos.",
    "Bancos de dados vetoriais armazenam embeddings e permitem buscas por similaridade, sendo muito utilizados em aplicações de RAG.",
    "Modelos de Question Answering recebem uma pergunta e um contexto, retornando uma resposta baseada no texto fornecido."
]

pd.DataFrame({"id": range(1, len(documentos) + 1), "texto": documentos})

## 4. Criação dos embeddings

Aqui usamos um modelo multilíngue do HuggingFace para transformar cada documento em um vetor numérico. Esses vetores serão usados para encontrar qual documento é mais parecido com a pergunta.

In [ ]:
modelo_embeddings = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings criados com sucesso!")
print("Quantidade de documentos:", len(documentos))
print("Dimensão de cada embedding:", len(embeddings_documentos[0]))

## 5. Função de busca do contexto mais relevante

A pergunta do usuário também é transformada em embedding. Depois calculamos a similaridade entre a pergunta e cada documento para recuperar o melhor contexto.

In [ ]:
def buscar_contexto(pergunta):
    embedding_pergunta = modelo_embeddings.encode([pergunta])
    similaridades = cosine_similarity(embedding_pergunta, embeddings_documentos)[0]
    indice_melhor = similaridades.argmax()

    return {
        "pergunta": pergunta,
        "contexto": documentos[indice_melhor],
        "similaridade": float(similaridades[indice_melhor])
    }

resultado_busca = buscar_contexto("O que é RAG?")
resultado_busca

## 6. Modelo de Question Answering

Agora carregamos um modelo do HuggingFace treinado para responder perguntas em português com base em um contexto.

In [ ]:
modelo_qa = pipeline(
    "question-answering",
    model="pierreguillou/bert-base-cased-squad-v1.1-portuguese",
    tokenizer="pierreguillou/bert-base-cased-squad-v1.1-portuguese"
)

print("Modelo de Question Answering carregado com sucesso!")

## 7. Função final do mini RAG

A função abaixo une as duas partes:

1. Recupera o contexto mais relevante;
2. Envia a pergunta e o contexto para o modelo de Question Answering;
3. Retorna a resposta final.

In [ ]:
def responder(pergunta):
    busca = buscar_contexto(pergunta)
    resposta = modelo_qa(question=pergunta, context=busca["contexto"])

    return {
        "pergunta": pergunta,
        "contexto_recuperado": busca["contexto"],
        "similaridade": round(busca["similaridade"], 3),
        "resposta": resposta["answer"],
        "confianca_modelo": round(resposta["score"], 3)
    }

## 8. Testes da aplicação

Execute as células abaixo para testar o sistema com diferentes perguntas.

In [ ]:
responder("O que é RAG?")

In [ ]:
responder("Para que servem embeddings?")

In [ ]:
responder("O que é o HuggingFace?")

## 9. Conclusão

A aplicação demonstrou um fluxo simplificado de RAG. Primeiro, o sistema recupera o trecho mais relevante da base de conhecimento. Depois, um modelo do HuggingFace responde com base nesse contexto.

Essa solução pode ser expandida para documentos maiores, PDFs, páginas da web ou bancos vetoriais, tornando as respostas mais contextualizadas e confiáveis.

## Referências

- HuggingFace: https://huggingface.co/
- Transformers: https://huggingface.co/docs/transformers
- Sentence Transformers: https://www.sbert.net/
- Modelo de embeddings: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Modelo de QA em português: https://huggingface.co/pierreguillou/bert-base-cased-squad-v1.1-portuguese